<style>
    .jp-RenderedHTMLCommon {
        font-family: "Times New Roman", Times, serif !important;
        font-size: 16px !important;
    }
</style>

### **1. Постановка задачи**

Дан интеграл вида

$$
\int\limits_a^b f(x)\,dx,
$$

где $a$, $b$ и $f(x)$ определены в задании №2 на тему «Приближение функций».

$$a, = 0.25, \quad b = 1.25$$

$$f(x) = 0.25 e^x + 0.75 \cos{x}$$

Для вычисления интеграла с точностью $\varepsilon = 10^{-5}$ необходимо:

1. Пользуясь выражением для погрешности интегрирования, определить шаг $h$ в составной квадратурной формуле, который обеспечит требуемую точность результата.  
Варианты СКФ на базе:  
- а) ЛП
- б) ПП
- **в) СП**
- г) Т
- д) Симпсон

2. Для СКФ из п.1 определить величину $h$ шага разбиения исходного отрезка интегрирования, достаточного для достижения точности $\varepsilon$, по правилу Рунге.

3. Применить квадратурную формулу НАСТ Гаусса при указанном значении $n$. Оценить погрешность интегрирования через формулу остаточного члена $R_n(f)$.
Варианты значений $n$:  
- а) 5  
- б) 4  
- **в) 3**
- г) 2  
- д) 1

4. Провести сравнительный анализ полученных в п.п.1–3 результатов.

---

### **2. Определение шага $h$ по формуле погрешности для составной формулы средних прямоугольников.**

Для варианта **в)** используется составная квадратурная формула средних прямоугольников:

$$
I_{cc}(f)= h\sum_{k=0}^{N-1} f\left( \frac{x_k + x_{k+1}}{2}\right)= h\sum_{k=0}^{N-1} f\left(a+\left(k+\frac12\right)h\right),
$$

где:
* $N$ - количество разбиений отрезка $[a, b]$,
* $h=(b-a)/N.$

Остаток составной формулы средних прямоугольников имеет вид:

$$
R_{cc}(f)= h^2 \frac{(b-a)}{24}f''(\eta) = \frac{(b-a)^3}{24N^2}f''(\eta), \quad \eta\in[a,b].
$$

Тогда для оценки погрешности используем неравенство:

$$|R_{cc}(f)| \leq \frac{(b-a)^3}{24N^2}M,$$

где:
$$M=\max_{x\in[a,b]}|f''(x)|$$

Найдём $f''(x)$:
$$f''(x)=0.25e^x-0.75\cos x.$$

Чтобы обеспечить точность $|R_{cc}(f)|\leq \varepsilon,$ достаточно выбрать $N$, удовлетворяющее условию:

$$ \frac{(b-a)^3}{24}M \leq \varepsilon.$$

Отсюда получаем:

$$N \leq \sqrt{\frac{(b-a)^3 M}{24\varepsilon}}.$$

$$N = \left\lceil \sqrt{\frac{(b-a)^3 M}{24\varepsilon}} \right\rceil$$

После нахождения допустимого значения $N$ выбираем $h$:

$$h = \frac{b-a}{N}.$$

In [1]:
import numpy as np
import pandas as pd

def f(x: float) -> float:
    return 0.25 * np.exp(x) + 0.75*np.cos(x)

def f_2_derivative(x: float) -> float:
    return 0.25 * np.exp(x) - 0.75*np.cos(x)

a = 0.25
b = 1.25
EPS = 1e-5

x = np.linspace(a, b, 1000)
M = np.max([np.abs(f_2_derivative(xi)) for xi in x])

N = int(np.ceil(np.sqrt ((M*(b-a)**3)/(24*EPS))))
h = (b-a)/N

print("N =", N)
print("h =", h)

N = 52
h = 0.019230769230769232


Вычислим приближённое значение интеграла и остатка по приведенным выше формулам. Практическую погрешность получим следующим образом:

$$ R = \left| I - I_{сп}\right|, $$


где $I$ - аналитическое решение исходного интеграла:

$$\int^{1.25}_{0.25} 0.25 e^x + 0.75 \cos x = \left. (0.25 e^x + 0.75  \sin x) \right|_{0.25}^{1.25} \approx 1.0777648802693225$$

In [ ]:
def midpoint_rule (a: float, h: float, N: int, f: callable) -> float:
    result = 0
    for i in range(N):
        result += f(a + (i + 0.5)*h)
    return result * h

integral_exact = 0.25 * (np.exp(b) - np.exp(a)) + 0.75 * (np.sin(b) - np.sin(a))
integral_approx = midpoint_rule(a, h, N, f)
r_theoretical = (M*(b-a)**3)/(24*N**2)
r_practical = abs(integral_approx - integral_exact)

midpoint_df = pd.DataFrame({
    "Значение": [
        integral_exact,
        integral_approx,
        r_theoretical,
        r_practical
    ]
},index=[
    "Точное значение интеграла",
    "Приближенное значение интеграла",
    "Теоретическая невязка",
    "Практическая невязка"
])

midpoint_df = midpoint_df.style.format({
    "Значение": lambda x:
        f"{x:.15f}" if x not in [r_theoretical, r_practical]
        else f"{x:.5e}"
})

midpoint_df

,Значение
Точное значение интеграла,1.077764880269322
Приближенное значение интеграла,1.077764489147145
Теоретическая невязка,9.80174e-06
Практическая невязка,3.91122e-07


### **3. Определение шага $h$ по правилу Рунге**

Для определения шага разбиения воспользуемся правилом Рунге. Пусть $I_h$ — приближённое значение интеграла, вычисленное составной квадратурной формулой с шагом $h$. Предположим, что остаток квадратурной формулы имеет разложение вида

$$
R(h,f)=ch^m+O(h^{m+1}),
$$

где $m$ — порядок точности квадратурной формулы.

Для составной формулы средних прямоугольников остаток имеет порядок

$$
R_{cc}(f)=O(h^2),
$$

следовательно, $m=2.$

Обозначим через $I_{h_1}$ и $I_{h_2}$ приближённые значения интеграла, вычисленные с шагами $h_1$ и $h_2$. Тогда главная часть погрешности определяется по формуле Рунге:

$$
R(h,f)\approx
\frac{I_{h_2}-I_{h_1}}
{1-\left(\frac{h_2}{h_1}\right)^m}.
$$

На практике обычно выбирают $h_2=\frac{h_1}{2}$, то есть шаг уменьшается в два раза, а количество разбиений $N_2 = 2N_1$ увеличивается вдвое. Тогда формула Рунге принимает вид:

$$
R(h,f)\approx
\frac{I_{h_2}-I_{h_1}}
{1-\left(\frac12\right)^m}.
$$

Будем последовательно увеличивать число разбиений $N$ до тех пор, пока не выполнится условие

$$
|R(h,f)|\leq \varepsilon.
$$

После этого шаг

$$
h_1=\frac{b-a}{N_1}
$$

будет считаться достаточным для достижения требуемой точности $\varepsilon$, а приближенное значение интеграла можно уточнить следующим образом:

$$I \approx I_{h_1} + \frac{I_{h_2} - I_{h_1}}{1-\left(\frac{h_2}{h_1}\right)^m}.

In [ ]:
def runge_midpoint_rule (a: float, b: float, EPS: float, f: callable, N_start: int = 1) -> tuple[float, float, int, float]:
    m = 2
    N1 = N_start
    N2 = 2*N1

    while True:
        h1 = (b-a)/N1
        integral_h1 = midpoint_rule (a, h1, N1, f)
        h2 = h1/2
  
        integral_h2 = midpoint_rule (a, h2, N2, f)
        denominator = (1 - (1/2)**m)
        residual = (integral_h2 - integral_h1) / denominator

        if abs(residual) < EPS:
            result = integral_h1 + residual
            return result, abs(residual), N1, h1
        N1, N2 = N2, 2*N2
        
integral_approx, r_theoretical, N, h = runge_midpoint_rule(a, b, EPS, f)
r_practical = abs(integral_approx - integral_exact)

runge_df = pd.DataFrame({
    "Значение": [
        integral_exact,
        integral_approx,
        r_theoretical,
        r_practical,
        N,
        h
    ]
},index=[
    "Точное значение интеграла",
    "Приближенное значение интеграла",
    "Теоретическая невязка",
    "Практическая невязка",
    "Количество разбиений",
    "Шаг разбиения"
])

runge_df = runge_df.style.format({
    "Значение": lambda x:
        f"{x:.5e}" if x in [r_theoretical, r_practical]
        else (
            f"{int(x)}" if x == N
            else (
                f"{x:g}" if x == h
                else f"{x:.15f}"
            )
        )
})

runge_df

,Значение
Точное значение интеграла,1.077764880269322
Приближенное значение интеграла,1.077764875272909
Теоретическая невязка,4.10814e-06
Практическая невязка,4.99641e-09
Количество разбиений,16
Шаг разбиения,0.0625


Полученное по правилу Рунге значение шага $h = 0.0625$ обеспечивает требуемую точность вычисления интеграла при использовании составной формулы средних прямоугольников. Для достижения заданной точности потребовалось $N = 16$ разбиений отрезка интегрирования.

---